# EDA of the rendered synthetic personal-assistant data

This notebook analyzes the existing rendered event file at `data/lifecycle_20261021_1318/02_rendered/messages/events_with_messages_rendered.jsonl`. It does **not** rewrite, clean, or regenerate that file. Persona/split/configuration context is read from the immutable raw-generation phase. Results are written to a new location, `outputs/eda_rendered/`, so the prior raw EDA artifacts remain untouched.

The corpus represents 36 fictional personas with 30 simulated days each. It is a synthetic benchmark, not real email or real personal data. The rendered file includes message events and preserves other event types. Some non-message events may still be pending text rendering; renderer failures remain explicit.

## Research basis and limitations

| Technique | Why used / issue detected | Metric | Supporting paper and limitation |
|---|---|---|---|
| Multi-axis fidelity, utility, privacy | Prevents one score from being misread as overall quality | Integrity, missingness, diversity, and privacy indicators kept separate | Lautrup et al., [Syntheval](https://doi.org/10.1007/s10618-024-01081-4); tabular measures need adaptation for text/sequences. |
| Rendered text and structured metadata checks | Detects empty, failed, repeated, or inconsistent rendered records | Render-status counts, text uniqueness, schema and referential checks | Ramesh et al., [SynthTextEval](https://aclanthology.org/2025.emnlp-demos.35/); automatic measures do not establish factual/natural quality, so human review remains necessary. |
| Subgroup coverage and fairness caution | Identifies sparse generator-defined cohorts without imposing identical preferences | Persona/event counts by synthetic locale, region, archetype, and renderer proxy | Nagesh et al., [FairCauseSyn](https://doi.org/10.1109/EMBC58623.2025.11252705); causal fairness requires a valid task-specific causal model and labels. |
| Temporal tail and split checks | Finds missing days, event-mix changes, and coverage collapse | Daily counts, autocorrelation, split-wise Jensen–Shannon divergence | Shumailov et al., [The Curse of Recursion](https://doi.org/10.1038/s41586-024-07566-y); warns about recursive synthetic training but does not prove this dataset is collapsed. |
| Downstream task evaluation | Utility is task-dependent, not guaranteed by distribution matching | Model training readiness only; held-out benchmark required | Tan et al., [PersonaBench](https://aclanthology.org/2025.findings-acl.49/); benchmark precedent, not a data-quality certificate. |

No real personal-data reference or real holdout is included. Real-versus-synthetic distances, membership inference against real people, and real-world generalization therefore remain unavailable. Regex/duplicate checks are not formal privacy guarantees.

## 1. Reproducible input selection

The notebook selects the rendered events file explicitly and uses the raw phase only for supporting metadata. It records source hashes before and after analysis, package versions, counts, output hashes, and a fixed analysis seed. No source is modified.

In [ ]:
from pathlib import Path
import json
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents]
                     if (p / "pyproject.toml").exists() and (p / "src" / "p17sim").is_dir()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path("/Users/bavishna/Documents/Bavishna_Masters/Sem 4/DATA298A/298A-Team-1/sandbox")
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Cannot locate the sandbox project root.")

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from synthetic_data_eda import discover_raw_dir, run_eda

RAW_METADATA_DIR = discover_raw_dir(PROJECT_ROOT)
RUN_DIR = PROJECT_ROOT / "data" / "lifecycle_20261021_1318"
RENDERED_EVENTS = RUN_DIR / "02_rendered" / "messages" / "events_with_messages_rendered.jsonl"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "eda_rendered"
if not RENDERED_EVENTS.is_file():
    raise FileNotFoundError(f"Rendered events file not found: {RENDERED_EVENTS}")
print(f"Rendered data (read-only): {RENDERED_EVENTS}")
print(f"Supporting raw metadata (read-only): {RAW_METADATA_DIR}")
print(f"Separate EDA outputs: {OUTPUT_DIR}")

## 2. Dataset overview and quality

Summarizes records, schema, data types, missingness, identifier uniqueness, day/minute ranges, persona references, split assignments, and potential numeric outliers. The IQR rule is exploratory; it can flag valid rare events. No rows are altered or removed.

In [ ]:
RESULT = run_eda(project_root_override=PROJECT_ROOT, raw_override=RAW_METADATA_DIR,
                 events_override=RENDERED_EVENTS, output_override=OUTPUT_DIR)
print(json.dumps(RESULT, indent=2))
overview = json.loads((OUTPUT_DIR / "statistics" / "dataset_overview.json").read_text())
for item in overview["generated_files"]:
    print(f"{item['file']}: {item['records']:,} rows, {item['features']} fields")

## 3. Event types, temporal patterns, and distribution shift

Daily/hourly event counts, rolling statistics, autocorrelation, daily event-type heatmaps, and split-wise event-type divergence reveal temporal gaps and composition changes. These are exploratory diagnostics on the generated cohort; a low divergence does not establish equivalence to real users.

In [ ]:
import pandas as pd
for filename in ["records_per_day.png", "daily_event_type_heatmap.png", "events_by_hour.png",
                 "daily_count_autocorrelation.png", "numeric_feature_correlation.png"]:
    path = OUTPUT_DIR / "figures" / filename
    if path.exists():
        print(f"Saved figure: {path}")
print(pd.read_csv(OUTPUT_DIR / "tables" / "integrity_checks.csv").to_string(index=False))
print(pd.read_csv(OUTPUT_DIR / "tables" / "daily_record_counts_and_rolling.csv").head(10).to_string(index=False))
print(pd.read_csv(OUTPUT_DIR / "tables" / "event_type_counts.csv").to_string(index=False))

## 4. Fidelity, bias, fairness, diversity, and privacy

Synthetic archetype/locale/region/name-list fields are generator proxies, not real demographic labels. This notebook reports support and balance but does not assert demographic parity or causal fairness. Exact/normalized text repetition is only a lexical signal; semantic diversity and human quality review remain open. The raw manifest contains no real personal-data comparator, so real-data Wasserstein/KS/JS tests, nearest-neighbor-to-real checks, and membership inference are not run. No differential privacy guarantee is claimed.

In [ ]:
for filename in ["persona_subgroup_counts.csv", "quality_issue_triage.csv",
                 "acceptance_criteria.csv", "adjacent_split_event_type_js.csv"]:
    path = OUTPUT_DIR / "tables" / filename
    if path.exists():
        print(f"\n{filename}\n{pd.read_csv(path).to_string(index=False)}")
for filename in ["diversity_coverage.json", "privacy_screening.json", "source_immutability.json"]:
    print(f"\n{filename}\n{(OUTPUT_DIR / 'statistics' / filename).read_text()}")

## 5. Downstream readiness and acceptance decision

Rendered events are still not model-specific training examples. Complete trace/label validation, preserve failed message records for review, render remaining file/fact-update text in a separate phase if required, then use train/validation personas only for training and keep test/OOD sealed. No downstream model was trained here, so utility, subgroup error, preference regret, deletion compliance, and real-world generalization are not claimed. Final report, CSV metrics, PNG figures, JSON statistics, citations, and execution log are saved under `outputs/eda_rendered/`.

In [ ]:
quality = json.loads((OUTPUT_DIR / "quality_report.json").read_text())
immutability = json.loads((OUTPUT_DIR / "statistics" / "source_immutability.json").read_text())
print(pd.read_csv(OUTPUT_DIR / "tables" / "acceptance_criteria.csv").to_string(index=False))
print(f"Raw/rendered sources modified: {quality['raw_data_modified']}")
print(f"Input hashes unchanged: {immutability['hashes_match_before_after']}")
print(f"Report: {OUTPUT_DIR / 'reports' / 'synthetic_data_eda_report.md'}")